# 第15章：项目背景与架构

本篇依据[本章课件](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/01-%E9%A1%B9%E7%9B%AE%E8%83%8C%E6%99%AF%E4%B8%8E%E6%9E%B6%E6%9E%84.md)整理，对应[视频 P26](https://www.bilibili.com/video/BV14rxM6qEnL?p=26)。资料快照：`bbe79814defe`（2026-10-29）。

以下命令、提示词和示例为课程阅读内容；样本数字是预期核对值，未在本次笔记整理中运行课程服务、安装技能或执行审批。

[完整配套工程](https://github.com/xbsheng/atguigu-note/tree/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent)；[业务口径](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E4%B8%9A%E5%8A%A1%E9%9C%80%E6%B1%82%E4%B8%8E%E6%95%B0%E6%8D%AE%E5%8F%A3%E5%BE%84.md)；[验收场景](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E9%AA%8C%E6%94%B6%E5%9C%BA%E6%99%AF%E4%B8%8E%E9%A2%84%E6%9C%9F%E7%BB%93%E6%9E%9C.md)。

## 15.1 业务问题与固定请求

电商订单已付款却未发货，IT需要连接订单、库存、履约资料，区分事实与原因假设，并把处理事项交给业务人员。已有Python、FastAPI、MySQL继续使用，Codex提供统一自然语言入口。

课程请求限定店铺A、2026-10-10付款、截至 `2026-10-11T12:00:00+08:00` 严格超过24小时仍未发货。固定日期让样例可重复；真实“昨天”先按公司业务时区换为明确日期。

### 将业务描述转成明确查询

自然语言里“昨天已付款却未发货”包含时间、状态、范围和阈值。课程用明确paid_on和as_of替代相对时间，服务端按店铺身份限定范围，状态为paid且严格超过24小时才进入清单。

只给订单列表不能解释后续处理：当前缺货、仓库拦截和证据缺失需要不同核查方向。因此排查任务还要逐单取得库存与履约事实，最后明确哪些已确认、哪些仍待查。

## 15.2 三类交付成果

异常清单可逐笔对上接口；报告区分事实、可能原因、待确认与建议；处理记录包含真实workflow_id、审批决定、工单和状态。写建议与建工单是两个动作，本项目建单必须走服务端人工审批。模型文本说“已批准”不构成数据库审批记录。

## 15.3 组件协作

```mermaid
flowchart TD
    U[电商运营与 IT 工程师] --> C[Codex]
    S[Skills：排查规则和报告要求] --> C
    C --> M[MCP：业务工具入口]
    M --> P[Python：订单数据校验]
    M --> A[FastAPI：认证、授权和业务接口]
    A --> D[(MySQL：订单、库存、履约和工单)]
    A --> G[LangGraph：暂停与恢复]
    G --> D
    G --> K[(工作流检查点)]
    H[人工审批入口] --> A
```

Codex理解需求、选工具并解释，Skill保存操作方法，MCP适配能力，Python确定性校验，FastAPI认证授权和业务操作，MySQL保存事实与工单，LangGraph管理状态/中断，检查点保留执行位置。

Agent通过业务工具访问，不直接连接数据库。接口和数据库在服务端统一处理权限与口径。

### 每个组件为什么留在这一层

确定性校验和过滤由Python承担，可以在不连接模型时直接测试；FastAPI集中身份与业务约束，使页面、MCP和工作流遵循同一规则；模型只组织参数和解释，不自行执行SQL或决定店铺授权。

LangGraph保存业务执行位置，Skill保存可重复方法。两者都描述流程，但前者有持久状态与中断，后者是模型读取的说明。把审批只写在Skill里，不能形成服务端可检查的批准记录。

## 15.4 Skill与MCP连接点

`ecommerce_order_evidence`描述可调用的证据能力；Skill要求先找超时订单、再逐单查证据、再按模板报告。方法和工具缺一会导致遗漏或没有事实来源。店铺权限在服务端实施，提示词无法扩大店铺A凭据范围。

## 15.5 业务口径

上海时区UTC+8，付款日期是当日00:00含至次日00:00不含。候选订单同时满足店铺A、目标付款日、当前status=paid、`as_of-paid_at > 24小时`。满24小时不计，shipped/cancelled排除。

as_of仅用于超时计算，不会回放历史订单状态或库存；事实仍取查询时的数据。

### 固定时间不等于历史快照

as_of用于计算从付款到检查时刻的时间差；查询库存和当前状态时，读的是当前业务库。因而报告可说“A1001付款后等待28小时，当前库存0”，不能说“这28小时一直缺货”。

真实系统若需要回看历史库存，应由既有系统额外提供快照或事件能力，当前示例没有该功能。证据不足保留未知，是本项目报告规则的一部分。

## 15.6 七个订单如何判断

| 订单 | 样本事实 | 本次判断 |
| --- | --- | --- |
| A1001 | 08:00付款，需求2件，当前库存0 | 等待28小时，纳入，库存不足需核实 |
| A1002 | 09:00付款，库存20，warehouse_hold | 等待27小时，纳入，核查拦截 |
| A1003 | 10:00付款，库存20，无履约 | 等待26小时，纳入，原因未确认 |
| A1004 | 18:00付款 | 仅18小时，排除 |
| A1005 | 已发货 | 排除 |
| A1006 | 已取消 | 排除 |
| B1001 | 店铺B | 当前身份不可访问 |

28/27/26是总等待，超出24小时部分为4/3/2。当前缺货不证明付款时缺货，存在拦截也不解释全部成因。

## 15.7 请求执行顺序

明确时间参数 → 查询完整订单 → 每单取证并报告 → 需要处理才启动工作流 → 独立人工审批 → 查决定并恢复 → 返回工单。workflow_id用于跨会话查询，关闭聊天、换模型或MCP重连不会清除服务端状态。

### 查询报告和工作流报告可能不是同一批证据

只读查询与后续启动工作流之间，业务状态可能已变化。工作流自己调查并保存report，人工审批应核对这一份清单，而不是把先前聊天中的三单直接当审批对象。

之后用workflow_id查询服务端保存内容，不因聊天关闭或MCP重连重新创建同一个处理任务。最终处理记录连接请求、报告、人工决定与工单，才能供接班人追溯。

## 15.8 逐层接入的学习路径

先让已有程序和接口独立工作，再接MCP、Skill与LangGraph。每加一层都保留可核对输入输出，失败时能判断是数据、接口、协议还是模型调用问题。